In [ ]:
import tkinter as tk
from tkinter import filedialog
from docx import Document
import pandas as pd
import re

def find_korean_in_word(file_path):
    """워드 파일 내 모든 한글 위치와 내용을 리스트로 반환"""
    doc = Document(file_path)
    results = []
    korean_pattern = re.compile('[가-힣ㄱ-ㅎㅏ-ㅣ]')
    
    for para_idx, para in enumerate(doc.paragraphs):
        line_text = para.text
        for char_idx, char in enumerate(line_text):
            if korean_pattern.match(char):
                results.append(
                    f"📄 Word 파일 [{file_path}]\n"
                    f"    → 문단 {para_idx+1}번, 문자 {char_idx+1}번\n"
                    f"    → 내용: '{line_text}'\n"
                    f"    → 발견 문자: '{char}'\n"
                )
    return results

def find_korean_in_excel(file_path):
    """엑셀 파일 내 모든 한글 위치와 내용을 리스트로 반환"""
    results = []
    korean_pattern = re.compile('[가-힣ㄱ-ㅎㅏ-ㅣ]')
    
    xls = pd.ExcelFile(file_path)
    for sheet_name in xls.sheet_names:
        sheet_df = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')
        
        for row_idx, row in sheet_df.iterrows():
            for col_idx, cell in enumerate(row):
                if pd.isna(cell):
                    continue
                cell_str = str(cell)
                for char_idx, char in enumerate(cell_str):
                    if korean_pattern.match(char):
                        results.append(
                            f"📊 Excel 파일 [{file_path}]\n"
                            f"    → 시트: {sheet_name}\n"
                            f"    → 위치: {row_idx+1}행 {col_idx+1}열\n"
                            f"    → 내용: '{cell_str}'\n"
                            f"    → 발견 문자: '{char}'\n"
                        )
    return results

def open_word_file():
    file_path = filedialog.askopenfilename(filetypes=[("Word Files", "*.docx")])
    if file_path:
        result_text.delete(1.0, tk.END)
        results = find_korean_in_word(file_path)
        if results:
            for res in results:
                result_text.insert(tk.END, res + "\n" + "-"*50 + "\n")
        else:
            result_text.insert(tk.END, "⚠️ 한글 문자가 발견되지 않았습니다.")

def open_excel_file():
    file_path = filedialog.askopenfilename(filetypes=[("Excel Files", "*.xlsx *.xls")])
    if file_path:
        result_text.delete(1.0, tk.END)
        results = find_korean_in_excel(file_path)
        if results:
            for res in results:
                result_text.insert(tk.END, res + "\n" + "-"*50 + "\n")
        else:
            result_text.insert(tk.END, "⚠️ 한글 문자가 발견되지 않았습니다.")

# GUI 설정
root = tk.Tk()
root.title("🔍 한글 탐지기 - 상세 위치 보고 시스템")
root.geometry("800x600")

frame = tk.Frame(root)
frame.pack(pady=20)

btn_style = {'font': ('Malgun Gothic', 12), 'width': 20, 'height': 2}
btn_word = tk.Button(frame, text="📝 워드 파일 분석", command=open_word_file, **btn_style)
btn_word.grid(row=0, column=0, padx=10)

btn_excel = tk.Button(frame, text="📊 엑셀 파일 분석", command=open_excel_file, **btn_style)
btn_excel.grid(row=0, column=1, padx=10)

result_text = tk.Text(root, wrap=tk.WORD, font=('Malgun Gothic', 10))
result_text.pack(pady=10, padx=20, fill=tk.BOTH, expand=True)

root.mainloop()